In [7]:
from pathlib import Path
import geopandas as gpd
import pandas as pd
import folium

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
BOUNDARIES_DIR = DATA_DIR / "boundaries"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"

tocs_path       = BOUNDARIES_DIR / "tocs" / "toc_full_street_miles.gpkg"
villages_path   = BOUNDARIES_DIR / "villages" / "village_full_street_miles.gpkg"

tocs_path, villages_path

(WindowsPath('c:/Users/arjav/DevSource/toc-performance-dashboard/data/boundaries/tocs/toc_full_street_miles.gpkg'),
 WindowsPath('c:/Users/arjav/DevSource/toc-performance-dashboard/data/boundaries/villages/village_full_street_miles.gpkg'))

In [8]:
tocs = gpd.read_file(tocs_path)
villages = gpd.read_file(villages_path)

In [9]:
# Reproject to WGS84 for web maps
tocs_web     = tocs.to_crs(4326).copy()
villages_web = villages.to_crs(4326).copy()

In [11]:
# Map time!

# --- Base: villages in grey for context ---

m_streets = villages_web.explore(
    style_kwds={
        "fillOpacity": 0.2,
        "weight": 0.3,
        "color": "grey",
    },
    tooltip=[
        "NAME",
        "intersect_acres",
        "street_miles_total",
        "street_miles_per_acre"
    ],
    name="Villages – Street Miles per Acre",
)

# --- TOC choropleth by transit mode share ---
tocs_web.explore(
    m=m_streets,
    column="street_miles_per_acre",
    cmap="Blues",
    legend=True,
    legend_kwds={"caption": "Street Miles per Acre (TOC)"},
    style_kwds={
        "fillOpacity": 0.7,
        "weight": 0.7,
        "color": "black",
    },
    tooltip=[
        "APPLICABIL",
        "intersect_acres",
        "street_miles_total",
        "street_miles_per_acre"
    ],
    name="TOCs – Street Miles per Acre",
)

folium.LayerControl(collapsed=False).add_to(m_streets)

streets_map_path = OUTPUTS_DIR / "tocs_street_miles_per_acre.html"
m_streets.save(streets_map_path)

print("Saved:", streets_map_path)

Saved: c:\Users\arjav\DevSource\toc-performance-dashboard\outputs\tocs_street_miles_per_acre.html
